# Debug SVD-Attention Compression
This notebook mirrors `compress.py` with `--decompose_method=svd_attention` to:
- Load model and tokenizer
- Run rank search
- Replace attention with `LlamaPaluAttention`
- Optionally save a Hugging Face-format checkpoint
- Verify modules and run a quick sanity pass


In [3]:
%load_ext autoreload
%autoreload 2
# Imports and setup
import os
import torch
import itertools
from loguru import logger
from tqdm import tqdm

from utils import set_seed, dump_to_huggingface_repos, load_model_and_tokenizer
from palu.rank_search import rank_search
from palu.decomposition import compress_model

# Notebook logging
logger.remove()
logger.add(lambda msg: print(msg, end=""), colorize=True, level="INFO")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cuda


In [4]:
# Parameters (mirror CLI)
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
CALIB_DATASET = "wikitext2"
PARAM_RATIO_TARGET = 0.7
SEARCH_METHOD = "fisher_uniform"  # [fisher, fisher_uniform, uniform]
HEAD_GROUP_SIZE = 4
DECOMPOSE_METHOD = "svd_attention"  # Key change
USE_CACHE = True
SEED = 42
DUMP_HF = False  # Set False to skip saving

print({
    "MODEL_ID": MODEL_ID,
    "CALIB_DATASET": CALIB_DATASET,
    "PARAM_RATIO_TARGET": PARAM_RATIO_TARGET,
    "SEARCH_METHOD": SEARCH_METHOD,
    "HEAD_GROUP_SIZE": HEAD_GROUP_SIZE,
    "DECOMPOSE_METHOD": DECOMPOSE_METHOD,
    "USE_CACHE": USE_CACHE,
    "SEED": SEED,
    "DUMP_HF": DUMP_HF,
})


{'MODEL_ID': 'meta-llama/Meta-Llama-3-8B-Instruct', 'CALIB_DATASET': 'wikitext2', 'PARAM_RATIO_TARGET': 0.7, 'SEARCH_METHOD': 'fisher_uniform', 'HEAD_GROUP_SIZE': 4, 'DECOMPOSE_METHOD': 'svd_attention', 'USE_CACHE': True, 'SEED': 42, 'DUMP_HF': False}


In [5]:
# Load model & tokenizer
set_seed(SEED)
logger.info("Loading model and tokenizer...")
# delete model and tokenizer
model, tokenizer = load_model_and_tokenizer(MODEL_ID)
# Avoid overriding device_map="auto" here
model.eval()
print(type(model))
print(model.config)


2025-08-25 08:20:40.374 | INFO     | __main__:<module>:3 - Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
LlamaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "meta-llama/Meta-Llama-3-8B-Instruct",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "topk_svd": 1.0,
  "torch_dtype": "float16",
  "transformers_version": "4.49.0",
  "use_cache": true,
  "vocab_size": 128256
}



In [6]:
# Run rank search (layer-wise ranks for k_proj/v_proj)
class Args:
    model_id = MODEL_ID
    calib_dataset = CALIB_DATASET
    calib_seqlen = 1024
    param_ratio_target = PARAM_RATIO_TARGET
    search_method = SEARCH_METHOD
    head_group_size = HEAD_GROUP_SIZE
    device = device
    use_cache = USE_CACHE
    seed = SEED
    decompose_method = "svd_attention"

args = Args()

In [7]:
# Run rank search
search_results, rank_sum, total_rank = rank_search(model, tokenizer, args)
print(f"Search complete: rank_sum={rank_sum}, total_rank={total_rank}")


2025-08-25 08:20:48.974 | INFO     | palu.rank_search:rank_search:87 - [Rank search] Do rank searching. Search method: fisher_uniform
2025-08-25 08:20:48.979 | INFO     | palu.data_utils:get_calib_data:18 - [Calib data] Load from cache/wikitext2_meta-llama_Meta-Llama-3-8B-Instruct_32_1024_3.pt
2025-08-25 08:20:48.992 | INFO     | palu.rank_search:calib_fisher_info:42 - [Fisher] Search cache_file=cache/meta-llama_Meta-Llama-3-8B-Instruct_calib_fisher_info.pt
2025-08-25 08:20:48.992 | INFO     | palu.rank_search:calib_fisher_info:45 - [Fisher] File cache/meta-llama_Meta-Llama-3-8B-Instruct_calib_fisher_info.pt exist.
2025-08-25 08:20:48.993 | INFO     | palu.rank_search:calib_fisher_info:46 - [Fisher] Load cache_file=cache/meta-llama_Meta-Llama-3-8B-Instruct_calib_fisher_info.pt


/home/xinj/palu/palu/data_utils.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  traindataset = torch.load(cache_file)
/home/xinj/palu/palu/rank_search.py:47: FutureWarn

2025-08-25 08:20:51.070 | INFO     | palu.rank_search:rank_search:228 - [Rank Search] KV-Cache Compression Ratio:  29.69%
Search complete: rank_sum=46080, total_rank=65536


In [8]:
# Compress with svd_attention (replace attention modules)
compress_model(model, tokenizer, args, device, search_results)
print("Attention replacement done.")


2025-08-25 08:20:51.119 | INFO     | palu.decomposition:compress_model:352 - Model compressing...
Attention replacement done.


In [9]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaPaluAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): HeadwiseLowRankModule(
            (VT): Linear(in_features=4096, out_features=320, bias=False)
            (U): ModuleList(
              (0-1): 2 x Linear(in_features=160, out_features=512, bias=False)
            )
          )
          (v_proj): HeadwiseLowRankModule(
            (VT): Linear(in_features=4096, out_features=1024, bias=False)
            (U): ModuleList(
              (0-1): 2 x Linear(in_features=512, out_features=512, bias=False)
            )
          )
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_

In [11]:
# Minimal perplexity sanity check (few tokens)
from datasets import load_dataset

@torch.no_grad()
def quick_ppl(model, tokenizer, seqlen=512, max_samples=2):
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(ds["text"])[:100000]
    enc = tokenizer(text, return_tensors="pt")
    input_ids = enc.input_ids[:, : seqlen * max_samples].to(model.device)
    nsamples = input_ids.numel() // seqlen
    nlls = []
    for i in range(nsamples):
        batch = input_ids[:, i*seqlen:(i+1)*seqlen]
        outputs = model.model(batch)
        hidden_states = outputs[0]
        logits = model.lm_head(hidden_states)
        shift_logits = logits[:, :-1, :]
        shift_labels = batch[:, 1:]
        loss = torch.nn.functional.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)), shift_labels.reshape(-1)
        )
        nlls.append(loss * seqlen)
    ppl = torch.exp(torch.stack(nlls).sum() / (len(nlls) * seqlen))
    return ppl.item()

try:
    ppl = quick_ppl(model, tokenizer, seqlen=512, max_samples=2)
    print("Quick PPL:", ppl)
except Exception as e:
    print("Sanity PPL failed:", e)


Quick PPL: 10.296875


In [ ]:
# Re-save to HF format with correct config
DUMP_HF = True  # Enable saving
if DUMP_HF:
    save_folder = f"{MODEL_ID.split('/')[-1]}_ratio-{PARAM_RATIO_TARGET}_gs-{HEAD_GROUP_SIZE}-{SEARCH_METHOD}-{DECOMPOSE_METHOD}"
    dump_to_huggingface_repos(model, tokenizer, save_folder, args)
    print("Re-saved to:", save_folder)

In [13]:
# Test loading the saved model
from transformers import AutoModelForCausalLM, AutoTokenizer
DUMP_HF=True
if DUMP_HF:
    save_folder = f"{MODEL_ID.split('/')[-1]}_ratio-{PARAM_RATIO_TARGET}_gs-{HEAD_GROUP_SIZE}-{SEARCH_METHOD}-{DECOMPOSE_METHOD}"
    print(f"Testing load of: {save_folder}")
    
    # Load the saved model
    loaded_model = AutoModelForCausalLM.from_pretrained(
        save_folder,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map="auto"
    )
    loaded_tokenizer = AutoTokenizer.from_pretrained(save_folder)
    
    print(f"Loaded model type: {type(loaded_model)}")
    print(f"Loaded config: decompose_method={getattr(loaded_model.config, 'decompose_method', 'None')}")
    
    # Check a few layers
    replaced_loaded = []
    for i, layer in enumerate(loaded_model.model.layers[:3]):
        replaced_loaded.append(type(layer.self_attn).__name__)
    print(f"First 3 attention types: {replaced_loaded}")
    
    # Quick sanity check
    try:
        test_input = loaded_tokenizer("Hello world", return_tensors="pt").to(loaded_model.device)
        with torch.no_grad():
            output = loaded_model(**test_input)
        print(f"Forward pass successful, logits shape: {output.logits.shape}")
    except Exception as e:
        print(f"Forward pass failed: {e}")
else:
    print("Skipping load test (DUMP_HF=False)")


Testing load of: Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-svd_attention
Replacing LlamaAttention with LlamaPaluAttention


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: Trying to set a tensor of shape torch.Size([512, 160]) in "weight" (which has shape torch.Size([256, 416])), this look incorrect.

In [ ]:
# Test the fixed loading logic
from transformers import AutoModelForCausalLM, AutoTokenizer

save_folder = f"{MODEL_ID.split('/')[-1]}_ratio-{PARAM_RATIO_TARGET}_gs-{HEAD_GROUP_SIZE}-{SEARCH_METHOD}-{DECOMPOSE_METHOD}"
print(f"Testing fixed load of: {save_folder}")

try:
    # Load the saved model
    loaded_modelb = AutoModelForCausalLM.from_pretrained(
        save_folder,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map="auto"
    )
    loaded_tokenizer = AutoTokenizer.from_pretrained(save_folder)
    
    print(f"✓ Loaded model type: {type(loaded_model)}")
    print(f"✓ Loaded config: decompose_method={getattr(loaded_model.config, 'decompose_method', 'None')}")
    
    # Check a few layers
    replaced_loaded = []
    for i, layer in enumerate(loaded_model.model.layers[:3]):
        replaced_loaded.append(type(layer.self_attn).__name__)
    print(f"✓ First 3 attention types: {replaced_loaded}")
    
    # Check k_proj structure for first layer
    layer0 = loaded_model.model.layers[0].self_attn
    if hasattr(layer0, 'k_proj') and hasattr(layer0.k_proj, 'ranks'):
        print(f"✓ Layer 0 k_proj ranks: {layer0.k_proj.ranks}")
        print(f"✓ Layer 0 k_proj VT shape: {layer0.k_proj.VT.weight.shape}")
        if hasattr(layer0.k_proj, 'U'):
            print(f"✓ Layer 0 k_proj U shapes: {[u.weight.shape for u in layer0.k_proj.U]}")
    
    # Quick sanity check
    test_input = loaded_tokenizer("Hello world", return_tensors="pt").to(loaded_model.device)
    with torch.no_grad():
        output = loaded_model(**test_input)
    print(f"✓ Forward pass successful, logits shape: {output.logits.shape}")
    
    # Quick PPL test
    ppl = quick_ppl(loaded_model, loaded_tokenizer, seqlen=512, max_samples=2)
    print(f"✓ Loaded model PPL: {ppl}")
    
    if ppl < 50:  # Reasonable PPL threshold
        print("🎉 Load test PASSED! PPL is reasonable.")
    else:
        print("❌ Load test FAILED! PPL is too high.")
    
except Exception as e:
    print(f"❌ Load test failed: {e}")
    import traceback
    traceback.print_exc()


Testing fixed load of: Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-svd_attention


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device device because they were offloaded to the cpu.


Post-load: Replacing LlamaAttention with LlamaPaluAttention
✓ Loaded model type: <class 'palu.model.svd_llama.modeling_palu_llama.PaluLlamaForCausalLM'>
✓ Loaded config: decompose_method=svd_attention
✓ First 3 attention types: ['LlamaPaluAttention', 'LlamaPaluAttention', 'LlamaPaluAttention']
✓ Layer 0 k_proj ranks: [160, 160]
✓ Layer 0 k_proj VT shape: torch.Size([320, 4096])
✓ Layer 0 k_proj U shapes: [torch.Size([512, 160]), torch.Size([512, 160])]
✓ Forward pass successful, logits shape: torch.Size([1, 3, 128256])
✓ Loaded model PPL: 10.296875
🎉 Load test PASSED! PPL is reasonable.


In [20]:
print(loaded_model)

PaluLlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaPaluAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): HeadwiseLowRankModule(
            (VT): Linear(in_features=4096, out_features=320, bias=False)
            (U): ModuleList(
              (0-1): 2 x Linear(in_features=160, out_features=512, bias=False)
            )
          )
          (v_proj): HeadwiseLowRankModule(
            (VT): Linear(in_features=4096, out_features=1024, bias=False)
            (U): ModuleList(
              (0-1): 2 x Linear(in_features=512, out_features=512, bias=False)
            )
          )
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, 